[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# asyncpg &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with `fresh` and `tally`. Run it first. Each task opens the
connections it needs and closes them, so they can be run in any order.


In [1]:
import asyncio
import getpass
import json
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from asyncpg import exceptions

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

async def fresh(conn):
    """An empty tally table, so the sections below can be run in any order."""
    await conn.execute("DROP TABLE IF EXISTS tally")
    await conn.execute("CREATE TABLE tally (kind text, n int)")


async def tally(conn):
    """What is in it, read through whichever connection is asking."""
    return [tuple(row) for row in await conn.fetch("SELECT kind, n FROM tally ORDER BY kind")]


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


**1.** One answer, three shapes.


In [2]:
conn = await asyncpg.connect(database="guide")

print("fetch:   ", await conn.fetch("SELECT count(*) FROM events"))
print("fetchrow:", await conn.fetchrow("SELECT count(*) FROM events"))
print("fetchval:", await conn.fetchval("SELECT count(*) FROM events"))
await conn.close()


fetch:    [<Record count=5000>]
fetchrow: <Record count=5000>
fetchval: 5000


A list of one `Record`, one `Record`, and the value. `fetchval` is the one to reach for when the
query returns a single number, and the other two are there for when it does not.


**2.** The kind as a parameter.


In [3]:
conn = await asyncpg.connect(database="guide")

for kind in ("click", "view", "purchase"):
    print(f"{kind:9}", await conn.fetchval(
        "SELECT count(*) FROM events WHERE kind = $1", kind))

await conn.close()


click     1666
view      1667
purchase  1667


One query, prepared once by the server and run three times with different values, which is what
**Prepared Statements** is about next.


**3.** A write with no block at all.


In [4]:
conn = await asyncpg.connect(database="guide")
watcher = await asyncpg.connect(database="guide")

await fresh(conn)
await conn.execute("INSERT INTO tally VALUES ($1, $2)", "written alone", 1)

print("the writer sees: ", await tally(conn))
print("the watcher too: ", await tally(watcher))
await conn.close()
await watcher.close()


the writer sees:  [('written alone', 1)]
the watcher too:  [('written alone', 1)]


No `commit` anywhere, and the row is visible from another session immediately. In psycopg the
watcher would have seen an empty table until the writer committed.


**4.** The same write, inside a block that fails.


In [5]:
conn = await asyncpg.connect(database="guide")
watcher = await asyncpg.connect(database="guide")

await fresh(conn)
try:
    async with conn.transaction():
        await conn.execute("INSERT INTO tally VALUES ($1, $2)", "written in a block", 1)
        print("the writer sees it inside the block:", await tally(conn))
        raise RuntimeError("and then something went wrong")
except RuntimeError as error:
    print("the block failed:", error)

print("the watcher sees:", await tally(watcher))
await conn.close()
await watcher.close()


the writer sees it inside the block: [('written in a block', 1)]
the block failed: and then something went wrong
the watcher sees: []


Three states worth separating: inside the block the writer's own session sees the row, the watcher
never does, and after the exception nobody does. That is what the block bought.


**5.** Half a second, and no more.


In [6]:
conn = await asyncpg.connect(database="guide")

start = time.perf_counter()
try:
    await conn.fetchval("SELECT pg_sleep(10)", timeout=0.5)
except TimeoutError:
    print(f"gave up after {time.perf_counter() - start:.1f}s, not 10")

print("still usable:", await conn.fetchval("SELECT 1"))
await conn.close()


gave up after 0.5s, not 10
still usable: 1


The exception carries no message, so catching it by class and saying what timed out yourself is the
only way a log entry will be readable.


**6.** A jsonb column as a dict.


In [7]:
conn = await asyncpg.connect(database="guide")
await conn.set_type_codec("jsonb", encoder=json.dumps, decoder=json.loads, schema="pg_catalog")

for record in await conn.fetch("SELECT id, payload FROM events ORDER BY id LIMIT 3"):
    print(record["id"], record["payload"], "| size:", record["payload"]["size"])

await conn.close()


1 {'n': 1, 'size': 2} | size: 2
2 {'n': 2, 'size': 3} | size: 3
3 {'n': 3, 'size': 4} | size: 4


The codec is per connection, so it has to be registered on every connection you open, which in
**Connection Pools** becomes the `init` argument the pool calls for you.


---

&#8592; **Back to:** [asyncpg](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/11-asyncpg.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
